# TECHIN 515 Lab 2: Model Compression for Edge Deployment

## Why Model Compression?

Deep learning models are powerful but large. A typical image classification model like EfficientNet has **~4 million parameters** stored as 32-bit floats, resulting in a model file of **~55 MB**. 

**The problem:** 
**Pick your favorite microcontroller, and report whether the chosen microcontroller is compatible with EfficientNet**. 

Your mission in the remainder of lab is to explore how much compression techniques can help ML models fit into microcontrollers -- and whether it's enough.

## Learning Objectives

By the end of this lab, you will be able to:
1. **Explain** how quantization reduces model size by changing weight precision
2. **Apply** Float-16, Dynamic Range, and Integer quantization to a trained model
3. **Implement** magnitude-based pruning with sparsity scheduling
4. **Analyze** accuracy vs. size trade-offs and create comparison visualizations
5. **Evaluate** whether compressed models fit ESP32 memory constraints

## Lab Overview

You will apply three **post-training quantization** methods and **magnitude-based pruning** to compress an EfficientNet model trained on the Cats vs. Dogs dataset. You will measure the size and accuracy of each compressed model and visualize the tradeoffs.

**What you will do:**
1. Train an EfficientNet classifier on cats vs. dogs (~97% accuracy)
2. Apply Float-16, Dynamic Range, and Integer quantization
3. Apply weight pruning with gradual sparsity scheduling
4. Write code to measure sizes, track results, and visualize tradeoffs
5. Evaluate whether any compressed model fits on ESP32

## Setup

You need to set up the `TECHIN515` virtual environment to run this lab. The environment file is provided in `environment.yml`.

```bash
conda env create -f environment.yml
conda activate TECHIN515
```

See the [conda documentation](https://docs.conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html) for help with environment management.

> **Note for macOS Apple Silicon users:** If you encounter TensorFlow installation issues, replace `tensorflow==2.7.0` with `tensorflow-macos==2.7.0` in `environment.yml`.

In the following we will install `tensorflow_datasets` and `tensorflow_model_optimization` libraries.

In [1]:
pip install tensorflow_datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install tensorflow_model_optimization

Note: you may need to restart the kernel to use updated packages.


Next we will first download and load `cats_vs_dogs` dataset, load and retrain `efnet` ML model, and experiment with three (post training) quantization methods: (1) Float-16 Quantization, (2) Dynamic Range Quantization, and (3) Integer Quantization.  

The following code and material were adapted from the reference [1].

In [1]:
# Importing necessary libraries and packages.
import os
import numpy as np
import tempfile
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
from tensorflow.keras.models import Model
import tensorflow_model_optimization as tfmot
from tensorflow.keras.layers import Dropout, Dense, BatchNormalization
import matplotlib.pyplot as plt

In [ ]:
def get_model_size(filepath: str) -> float:
    """Return model file size in MB.

    TODO: Implement this function.
    Hint: Use os.path.getsize() to get the file size in bytes,
    then convert to megabytes (1 MB = 1024 * 1024 bytes).
    Return the result rounded to 2 decimal places using round().
    """
    # YOUR CODE HERE
    pass

# Results tracker -- you'll add entries after each compression technique.
# At the end of the lab, you'll use this to create a comparison visualization.
results = {}

## (1) Preparing the Dataset

We can directly import the dataset from the TensorFlow Dataset (tfds). Here we will split the dataset into training, validation, and testing set with a split ratio of 0.7:0.2:0.1. The as_supervised parameter is kept True as we need the labels of the images for classification. 

In [ ]:
# Downloading and Loading the CatvsDog dataset.
(train_ds, val_ds, test_ds), info = tfds.load('cats_vs_dogs', split=['train[:70%]', 'train[70%:90%]', 'train[90%:]'], shuffle_files=True, as_supervised=True, with_info=True)


Let us now have a look at the dataset information provided in tfds.info(). The dataset has two classes labeled as ‘cat’ and ‘dog’ with 16283, 4653, 2326 training, validation and testing images.

In [ ]:
# Obtaining dataset information.
print("Number of  Classes: " + str(info.features['label'].num_classes))
print("Classes : " + str(info.features['label'].names))
NUM_TRAIN_IMAGES = tf.data.experimental.cardinality(train_ds).numpy()
print("Training Images: " + str(NUM_TRAIN_IMAGES))
NUM_VAL_IMAGES = tf.data.experimental.cardinality(val_ds).numpy()
print("Validation Images: " + str(NUM_VAL_IMAGES))
NUM_TEST_IMAGES = tf.data.experimental.cardinality(test_ds).numpy()
print("Testing Images: " + str(NUM_TEST_IMAGES))

The function `tfds.visualization.show_examples()` function displays images and their corresponding labels. It comes very handy when we want to visualize a few images in a single line of code!

In [ ]:
# Visualizing the training dataset.
vis = tfds.visualization.show_examples(train_ds, info)

We have chosen 16 as batch size and 224×224 as image size so that the dataset can be processed effectively and efficiently. To prepare the dataset, the images have been resized accordingly.
Let’s also make sure to use buffered prefetching to yield data from the disk. Prefetching overlaps the preprocessing and model execution of a training step. Doing so reduces the step time to the training and the time it takes to extract the data.

In [ ]:
# Defining batch-size and input image size.
batch_size = 16
img_size = [224, 224]# Resizing images in the dataset.
train_ds_ = train_ds.cache().map(lambda x, y: (tf.image.resize(x, img_size), y)).batch(batch_size).prefetch(buffer_size=10)
val_ds_ = val_ds.cache().map(lambda x, y: (tf.image.resize(x, img_size), y)).batch(batch_size).prefetch(buffer_size=10)
test_ds_ = test_ds.cache().map(lambda x, y: (tf.image.resize(x, img_size), y)).batch(batch_size).prefetch(buffer_size=10)

To feed images to the TF Lite model, we need to extract the test images and their labels. We will store them into variables and feed them to TF Lite for evaluation.

In [ ]:
# Extracting and saving test images and labels from the test dataset.
test_images = []
test_labels = []
for image, label in test_ds_.take(len(test_ds_)).unbatch():
    test_images.append(image)
    test_labels.append(label)

## (2) Loading the Model

We have chosen the EfficientNet B0 model pre-trained on the ImageNet dataset for image classification purposes. EfficientNet is a state-of-the-art image classification model that significantly outperforms other ConvNets of similar size.

We import the model from `tf.keras.applications()`. The last layer has been removed by setting `include_top=False`. We have set the input image size to 224x224 pixels and kept the pooling layer as GlobalMaxPooling2D. All layers are unfrozen to make them trainable for fine-tuning.

In [ ]:
# Defining the model architecture.
efnet = tf.keras.applications.EfficientNetB0(include_top = False, weights ='imagenet', input_shape = (224, 224, 3), pooling = 'max')

# Unfreezing all the layers of the model for fine-tuning.
for layer in efnet.layers:
    layer.trainable = True

Now, we will add a Dense layer to the pre-trained model and train it. This layer will become the last layer, or the inference layer. We will also add Dropout and BatchNormalization to reduce overfitting.

In [ ]:
# Adding Dense, BatchNormalization and Dropout layers to the base model.
x = Dense(512, activation='relu')(efnet.output)
x = BatchNormalization()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.2)(x)
predictions = Dense(2, activation='softmax')(x)

## (3) Compiling the Model

We are ready to compile the model. We have used Adam Optimizer with an initial learning rate of 0.0001, sparse categorical cross-entropy as the loss function, and accuracy as the metric. Once compiled, we check the model summary.

In [ ]:
# Defining the input and output layers of the model.
model = Model(inputs=efnet.input, outputs=predictions)
 
# Compiling the model.
model.compile(optimizer=tf.keras.optimizers.Adam(0.0001), loss =tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False), metrics = ["accuracy"])
 
# Obtaining the model summary.
model.summary()

We are using Model Saving Callback and the Reduce LR Callback.

(i) Model Saving Callback saves the model with the best validation accuracy

(ii) Reduce LR Callback reduces the learning rate by a factor of 0.1 if validation loss remains the same for three consecutive epochs.

In [ ]:
# Defining file path of the saved model.
filepath = './model.h5'
 
# Defining Model Save Callback and Reduce Learning Rate Callback.
model_save = tf.keras.callbacks.ModelCheckpoint(
    filepath,
    monitor="val_accuracy",
    verbose=0,
    save_best_only=True,
    save_weights_only=False,
    mode="max",
    save_freq="epoch")
 
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.1, patience=3, verbose=1, min_delta=5*1e-3,min_lr =5*1e-9,)
 
callback = [model_save, reduce_lr]

### Discussion: Model Architecture

1. The model has ~4 million parameters stored as FP32 (4 bytes each). **Calculate** the theoretical model size in MB from the parameter count alone. Does the saved `model.h5` file match your calculation? If not, what else might be stored in the `.h5` file?

2. Based on the model summary, identify two different types of layers used in the architecture. What role does each play?

## (4) Training the Model

The method `model.fit()` is called to train the model. We pass the training and validation datasets and train the model for 15 epochs.

In [ ]:
# Training the model.
# ---> epoch = 3 for testing the code and saving time during the lab
# ---> epoch = 15 recommended for better results
model.fit(train_ds_, epochs=3, validation_data=val_ds_, shuffle=False, callbacks=callback)

## (5) Evaluating the Model

Done training! Let's check the model's performance on the test set, then convert to TFLite as a baseline for size comparison.

In [ ]:
# Evaluating the model on the test dataset.
_, baseline_model_accuracy = model.evaluate(test_ds_, verbose=0)

print(f'Baseline test accuracy: {baseline_model_accuracy * 100:.2f}%')
print(f'Baseline model size (.h5): {get_model_size("./model.h5")} MB')

In [ ]:
# Convert baseline model to TFLite WITHOUT quantization.
# This gives us a fair baseline: .h5 includes optimizer state, metadata, etc.
# TFLite strips those, so comparing .h5 to .tflite is not apples-to-apples.
baseline_converter = tf.lite.TFLiteConverter.from_keras_model(model)
baseline_tflite = baseline_converter.convert()

with open('./baseline_model.tflite', 'wb') as f:
    f.write(baseline_tflite)

baseline_tflite_size = get_model_size('./baseline_model.tflite')
baseline_h5_size = get_model_size('./model.h5')

print(f'Baseline Keras model (.h5):                {baseline_h5_size} MB')
print(f'Baseline TFLite model (no quantization):    {baseline_tflite_size} MB')
print(f'TFLite conversion alone reduced size by:    {(1 - baseline_tflite_size/baseline_h5_size)*100:.1f}%')

# Track baseline results
results['baseline_h5'] = {
    'size_mb': baseline_h5_size,
    'accuracy': baseline_model_accuracy * 100,
    'technique': 'Keras .h5 (original)'
}
results['baseline_tflite'] = {
    'size_mb': baseline_tflite_size,
    'accuracy': baseline_model_accuracy * 100,
    'technique': 'TFLite (no quantization)'
}

### Discussion: Training

1. Report your model's accuracy at 3 epochs. **Predict** what accuracy you'd get at 15 epochs, given that EfficientNet is pre-trained on ImageNet. If time permits, verify your prediction. Was the difference large or small? Explain why.

2. At what point does additional training stop improving accuracy? Why does this happen? (Hint: think about what "fine-tuning" a pre-trained model means -- the weights already encode useful features from ImageNet.)

## (6) Float-16 Quantization

In Float-16 quantization, weights are converted from 32-bit to 16-bit floating-point values, halving the storage per weight.

### Predict Before You Run

Before running the next cell, make your predictions:

- **Size prediction:** Each weight goes from 4 bytes (FP32) to 2 bytes (FP16). What compression ratio do you expect? Predicted size relative to baseline TFLite: ___ MB
- **Accuracy prediction:** FP16 still has a wide numeric range. Will accuracy change significantly? Why or why not?

*Write your predictions here, then run the code to check.*

In [ ]:
# Passing the Keras model to the TF Lite Converter.
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Using float-16 quantization.
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

# Converting the model.
tflite_fp16_model = converter.convert()

# Saving the model.
with open('./fp_16_model.tflite', 'wb') as f:
    f.write(tflite_fp16_model)

# Measure and display size.
fp16_size = get_model_size('./fp_16_model.tflite')
print(f'FP16 model size: {fp16_size} MB')

We have passed the Float 16 quantization the `converter.target_spec.supported_type` to specify the type of quantization. The rest of the code remains the same for a general way of conversion for the TF Lite Model. In order to get model accuracy, let’s first define evaluate() function that takes in tflite model and returns model accuracy.

In [ ]:
#Function for evaluating TF Lite Model over Test Images
def evaluate(interpreter):
    prediction= []
    input_index = interpreter.get_input_details()[0]["index"]
    output_index = interpreter.get_output_details()[0]["index"]
    input_format = interpreter.get_input_details()[0]['dtype']
    
    for i, test_image in enumerate(test_images):
        if i % 100 == 0:
            print('Evaluated on {n} results so far.'.format(n=i))
        test_image = np.expand_dims(test_image, axis=0).astype(input_format)
        interpreter.set_tensor(input_index, test_image)

        # Run inference.
        interpreter.invoke()
        output = interpreter.tensor(output_index)
        predicted_label = np.argmax(output()[0])
        prediction.append(predicted_label)
    
    print('\n')
    # Comparing prediction results with ground truth labels to calculate accuracy.
    prediction = np.array(prediction)
    accuracy = (prediction == test_labels).mean()
    return accuracy

Check this FP-16 Quantized TF Lite’s model performance on the Test Set. 

In [ ]:
# Passing the FP-16 TF Lite model to the interpreter.
interpreter = tf.lite.Interpreter('./fp_16_model.tflite')
# Allocating tensors.
interpreter.allocate_tensors()
# Evaluating the model on the test dataset.
test_accuracy = evaluate(interpreter)

print(f'Float 16 Quantized TFLite Model Test Accuracy: {test_accuracy * 100:.2f}%')
print(f'Baseline Keras Model Test Accuracy: {baseline_model_accuracy * 100:.2f}%')

# Track results
results['fp16'] = {
    'size_mb': get_model_size('./fp_16_model.tflite'),
    'accuracy': test_accuracy * 100,
    'technique': 'Float-16 Quantization'
}

### Discussion: Float-16 Quantization

1. Calculate the **theoretical compression ratio** for FP32 to FP16 (hint: how many bytes per weight?). Does the actual file size reduction match your calculation? If not, explain why (hint: not all data in a model file is weights).

2. Were your predictions correct? FP16 accuracy is nearly identical to baseline. Why is FP16 quantization considered practically "lossless" for most models? Under what conditions might it cause noticeable accuracy loss?

## (7) Dynamic Range Quantization

In Dynamic Range Quantization, weights are converted to 8-bit integer precision, but **activations remain in floating point** and are dynamically quantized at inference time. This "mixed precision" approach often preserves accuracy better than full integer quantization.

Note: The converter will skip quantization for layers with fewer than 1024 elements, as the overhead of quantization outweighs the size savings for very small layers.

### Predict Before You Run

- **Size prediction:** Weights go from FP32 (4 bytes) to INT8 (1 byte). What compression ratio do you expect? But remember that activations stay in float and not all layers are quantized. Predicted size: ___ MB
- **Accuracy prediction:** Weights lose more precision than in FP16. Will accuracy drop? By how much?

In [ ]:
# Dynamic Range Quantization
# Weights are quantized to INT8; activations remain in float and are
# dynamically quantized at inference time.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_model = converter.convert()

with open('./dynamic_quant_model.tflite', 'wb') as f:
    f.write(tflite_quant_model)

dynamic_size = get_model_size('./dynamic_quant_model.tflite')
print(f'Dynamic Range model size: {dynamic_size} MB')

Let’s evaluate this TF Lite model on the test dataset.

In [ ]:
# Passing the Dynamic Range Quantized TF Lite model to the Interpreter.
interpreter = tf.lite.Interpreter('./dynamic_quant_model.tflite')
# Allocating tensors.
interpreter.allocate_tensors()
# Evaluating the model on the test images.
test_accuracy = evaluate(interpreter)

print(f'Dynamic Range Quantized TFLite Model Test Accuracy: {test_accuracy * 100:.2f}%')
print(f'Dynamic Range Model Size: {get_model_size("./dynamic_quant_model.tflite")} MB')
print(f'Baseline Keras Model Test Accuracy: {baseline_model_accuracy * 100:.2f}%')

# Track results
results['dynamic'] = {
    'size_mb': get_model_size('./dynamic_quant_model.tflite'),
    'accuracy': test_accuracy * 100,
    'technique': 'Dynamic Range Quantization'
}

### Discussion: Dynamic Range Quantization

1. Dynamic range quantization converts weights to INT8 but keeps activations in float. Why might this "mixed precision" approach preserve accuracy better than quantizing everything to INT8?

2. The converter log showed it skipped quantization for layers with fewer than 1024 elements. Why would quantizing very small layers be counterproductive? (Think about the overhead of storing scale/zero-point metadata vs. the savings from fewer bits.)

3. Were your size and accuracy predictions correct? Explain any discrepancies.

## (8) Integer Quantization

Integer quantization is the most aggressive of the three methods: it converts **both weights AND activations** to 8-bit integers. This results in a smaller model and increased inferencing speed, which is especially valuable for microcontrollers.

Unlike FP16 and dynamic range quantization, integer quantization requires a **representative dataset** -- a small sample of typical inputs used for **calibration**. The converter uses these samples to run inference and observe the range of activation values at each layer, so it can map floating-point activations to INT8 without clipping important values.

> **Important for ESP32 deployment:** We use only `TFLITE_BUILTINS_INT8` as the supported ops set. The alternative `SELECT_TF_OPS` requires a full TensorFlow runtime, which is **not available** on microcontrollers like the ESP32. Only `TFLITE_BUILTINS_INT8` is compatible with TensorFlow Lite Micro.

### Predict Before You Run

- **Size prediction:** Both weights and activations are INT8. Will the model be smaller than dynamic range quantization? Why or why not?
- **Accuracy prediction:** Now activations are also quantized. Do you expect a bigger accuracy drop than the previous methods?

In [ ]:
# Integer Quantization
# Both weights AND activations are quantized to INT8.
# A representative dataset is needed for activation calibration.

def representative_data_gen():
    """Yield sample inputs for INT8 calibration."""
    for input_value in tf.data.Dataset.from_tensor_slices(test_images).batch(1).take(100):
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8
tflite_int_model = converter.convert()

with open('./int_quant_model.tflite', 'wb') as f:
    f.write(tflite_int_model)

int8_size = get_model_size('./int_quant_model.tflite')
print(f'Integer Quantized model size: {int8_size} MB')

Let’s evaluate the obtained Integer Quantized TF Lite model on the test dataset.

In [ ]:
# Passing the Integer Quantized TF Lite model to the Interpreter.
interpreter = tf.lite.Interpreter('./int_quant_model.tflite')
# Allocating tensors.
interpreter.allocate_tensors()
# Evaluating the model on the test images.
test_accuracy = evaluate(interpreter)

print(f'Integer Quantized TFLite Model Test Accuracy: {test_accuracy * 100:.2f}%')
print(f'Integer Quantized Model Size: {get_model_size("./int_quant_model.tflite")} MB')
print(f'Baseline Keras Model Test Accuracy: {baseline_model_accuracy * 100:.2f}%')

# Track results
results['int8'] = {
    'size_mb': get_model_size('./int_quant_model.tflite'),
    'accuracy': test_accuracy * 100,
    'technique': 'Integer Quantization (INT8)'
}

### Discussion: Integer Quantization

1. Integer quantization dropped accuracy by ~5% while FP16 had almost no drop. **Explain WHY** integer quantization causes more accuracy loss. What additional step does it perform (hint: activations) that the other methods do not?

2. The code uses a "representative dataset" of 100 images for calibration. In your own words, explain what calibration does and why it is needed for integer quantization but not for FP16.

3. What would happen if your representative dataset contained **only cat images**? How might this affect accuracy on dog images? Why?

## (9) Model Pruning

Pruning takes a different approach from quantization: instead of reducing precision, it **sets unimportant weights to zero**. Specifically, **magnitude-based pruning** removes weights with the smallest absolute values -- the intuition being that near-zero weights contribute least to the model's output.

We use a **polynomial decay sparsity schedule**: starting at 50% sparsity (half the weights are zero) and gradually increasing to 80% sparsity over training. Pruning is applied only to Dense layers.

### Predict Before You Run

- **Size prediction:** If 80% of weights are zero, will the model file be 80% smaller? Why or why not? (Hint: think about how zeros are stored in a file vs. in memory)
- **Accuracy prediction:** Removing 80% of weights sounds dramatic. How much accuracy loss do you expect?

The following code and material were adapted from reference [2].

In [ ]:
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

# Compute end step to finish pruning after the specified epochs.
pruning_epochs = 2
num_batches_per_epoch = len(train_ds_)  # already batched, gives number of batches
end_step = num_batches_per_epoch * pruning_epochs

print(f"Pruning schedule: {end_step} total steps "
      f"({num_batches_per_epoch} batches/epoch x {pruning_epochs} epochs)")

# Define model for pruning.
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.50,
        final_sparsity=0.80,
        begin_step=0,
        end_step=end_step
    )
}

def apply_pruning_to_dense(layer):
    if isinstance(layer, tf.keras.layers.Dense):
        return tfmot.sparsity.keras.prune_low_magnitude(layer, **pruning_params)
    return layer

# Use tf.keras.models.clone_model to apply pruning to Dense layers.
model_for_pruning = tf.keras.models.clone_model(
    model,
    clone_function=apply_pruning_to_dense,
)

# prune_low_magnitude requires a recompile.
# Note: from_logits=False because the model's output layer uses softmax activation.
model_for_pruning.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

model_for_pruning.summary()

### Discussion: Pruning Configuration

1. **Experiment with different sparsity values.** Try at least 3 configurations and fill in this table:

| Initial Sparsity | Final Sparsity | Test Accuracy (%) | Model Size (MB) |
|:-:|:-:|:-:|:-:|
| 0.30 | 0.60 | | |
| 0.50 | 0.80 | | |
| 0.50 | 0.90 | | |

> **How to run each experiment:** Modify `initial_sparsity` and `final_sparsity` in the pruning configuration cell above, then re-run all cells from that cell through the pruning export/evaluation cell.

2. At what sparsity level does accuracy degrade significantly? Why?

3. The pruning schedule uses "polynomial decay" to gradually increase sparsity. Why is gradual pruning better than setting 80% of weights to zero all at once?

### Why the Number of Parameters Increases After Pruning

You may notice that the model summary shows **more** total parameters after applying pruning. This seems counterintuitive -- isn't pruning supposed to *reduce* parameters?

Here's what's happening: TensorFlow's pruning implementation adds a binary **mask tensor** for each pruned layer. These masks consist of 0s (pruned weights) and 1s (kept weights), and appear as **non-trainable parameters** in the summary.

**During training:** `output = (weight * mask) @ input` -- the mask zeros out pruned connections.

**After `strip_pruning()`:** The masks are removed, and pruned weights are permanently set to zero. The resulting model has the same architecture as the original, but many weights are zero, which enables more efficient compression when converting to TFLite.

The key insight: pruning doesn't remove parameters from the architecture -- it makes them zero, creating a **sparse** model that compresses better.

In [ ]:
# Extracting and saving test images and labels from the test dataset.
train_images = []
train_labels = []
for image, label in train_ds_.take(len(train_ds_)).unbatch():
    train_images.append(image)
    train_labels.append(label)

In [ ]:
logdir = tempfile.mkdtemp()

callbacks = [
    tfmot.sparsity.keras.UpdatePruningStep(),
    tfmot.sparsity.keras.PruningSummaries(log_dir=logdir),
]

model_for_pruning.fit(
    train_ds_,
    epochs=pruning_epochs,
    validation_data=val_ds_,
    shuffle=False,
    callbacks=callbacks
)

In [ ]:
_, model_for_pruning_accuracy = model_for_pruning.evaluate(test_ds_, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy) 
print('Pruned test accuracy:', model_for_pruning_accuracy)

In [ ]:
# Strip the pruning wrappers to get a standard Keras model.
model_for_export = tfmot.sparsity.keras.strip_pruning(model_for_pruning)

# Convert the pruned model to TFLite with dynamic range quantization.
converter = tf.lite.TFLiteConverter.from_keras_model(model_for_export)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
pruned_tflite_model = converter.convert()

# Save the pruned TFLite model.
with open('./pruned_model.tflite', 'wb') as f:
    f.write(pruned_tflite_model)

# Evaluate the pruned TFLite model.
interpreter = tf.lite.Interpreter('./pruned_model.tflite')
interpreter.allocate_tensors()
pruned_tflite_accuracy = evaluate(interpreter)

pruned_size = get_model_size('./pruned_model.tflite')
print(f'Pruned + Quantized TFLite Model Test Accuracy: {pruned_tflite_accuracy * 100:.2f}%')
print(f'Pruned TFLite Model Size: {pruned_size} MB')
print(f'Baseline Keras Model Test Accuracy: {baseline_model_accuracy * 100:.2f}%')

# Track results
results['pruned'] = {
    'size_mb': pruned_size,
    'accuracy': pruned_tflite_accuracy * 100,
    'technique': 'Pruning + Dynamic Quantization'
}

In [ ]:
# --- Results Comparison Table ---
# TODO: Build a formatted comparison table from the results dictionary.
# Print a table with columns: Technique, Size (MB), Accuracy (%), Compression Ratio, Fits ESP32?
# Hint: Loop over results.items() and use f-strings for formatting.
# The compression ratio = baseline_tflite_size / model_size
# A model fits ESP32 if its size <= 2.5 MB (4 MB flash - 1.5 MB firmware overhead)

# YOUR CODE HERE


In [ ]:
# --- Comparison Visualization ---
# TODO: Create a visualization comparing all compression methods.
# Use the `results` dictionary you've been building throughout the lab.
#
# Requirements:
# - Create a figure with 1 row, 2 columns (use plt.subplots)
# - Left: Horizontal bar chart of model sizes (MB). Add a vertical line at 2.5 MB
#   for the ESP32 budget. Label the line "ESP32 budget".
# - Right: Scatter plot of accuracy (%) vs. size (MB) for each model.
#   Add a vertical line at 2.5 MB for the ESP32 budget.
#   Annotate each point with the technique name.
#
# Useful functions: ax.barh(), ax.scatter(), ax.annotate(), ax.axvline(),
#                   ax.set_xlabel(), ax.set_title(), ax.legend()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Extract data from results dictionary
methods = [results[key]['technique'] for key in results]
sizes = [results[key]['size_mb'] for key in results]
accuracies = [results[key]['accuracy'] for key in results]

# YOUR CODE HERE: Create bar chart on ax1 (sizes) and scatter plot on ax2 (accuracy vs size)


plt.tight_layout()
plt.savefig('compression_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# --- ESP32 Feasibility Analysis ---
ESP32_FLASH_MB = 4.0
FIRMWARE_OVERHEAD_MB = 1.5
AVAILABLE_MB = ESP32_FLASH_MB - FIRMWARE_OVERHEAD_MB

print(f"\n{'='*60}")
print(f"ESP32 Feasibility Analysis")
print(f"{'='*60}")
print(f"Total flash: {ESP32_FLASH_MB} MB")
print(f"Firmware overhead: ~{FIRMWARE_OVERHEAD_MB} MB")
print(f"Available for model: ~{AVAILABLE_MB} MB")
print(f"{'='*60}")

for key, info in results.items():
    fits = "YES" if info['size_mb'] <= AVAILABLE_MB else "NO"
    print(f"{info['technique']:40s} | {info['size_mb']:6.2f} MB | Fits? {fits}")

### Discussion: Comprehensive Comparison

Review your `results` dictionary and visualization, then answer:

1. **Create a summary table.** Which method achieved the best compression ratio relative to the baseline TFLite model? Which had the worst accuracy trade-off?

2. **ESP32 feasibility:** Do any of your compressed models fit within the ESP32's flash memory (4 MB total, ~2.5 MB available after firmware overhead)? Show the calculation for each model.

3. **Trade-off analysis:** Integer quantization achieved significant compression but had the largest accuracy drop. Is a ~5% accuracy loss acceptable? Under what real-world conditions would it be acceptable vs. unacceptable?

4. **Key takeaway:** None of the compressed EfficientNet models fit on ESP32. What does this tell you about the relationship between model architecture choice and compression? If you were deploying a classifier on ESP32, what strategy would you pursue? Consider:
   - Using a different, inherently smaller model architecture
   - Combining multiple compression techniques
   - Upgrading to a more capable microcontroller
   - Justify your recommendation with data from this lab.

## When Compression Isn't Enough

As you've seen, even aggressive compression (INT8 quantization + pruning) couldn't shrink EfficientNet to fit on an ESP32. The compressed models are still several MB -- above the ~2.5 MB budget after firmware overhead.

This reveals a fundamental lesson in embedded ML:

**Compression is not a substitute for architecture selection.**

When deploying to severely constrained devices, you have three strategies:

1. **Better compression**: Quantization-aware training (QAT), structured pruning, knowledge distillation
2. **Smaller architecture**: Use models designed for embedded -- MobileNet, decision trees, tiny CNNs
3. **Different hardware**: Upgrade to a device with more memory (but this increases cost per unit)

In **Lab 3**, you'll take strategy #2: training a decision tree classifier that's only ~10 KB -- small enough to fit on ESP32 with room to spare. In **Lab 5**, you'll explore a hybrid edge-cloud approach where the ESP32 handles easy predictions locally and offloads difficult ones to the cloud.

## Additional Challenges (Optional)

If you finish early, try these challenges:

1. **Pruning + INT8 Quantization**: Apply integer quantization (instead of dynamic range) to the pruned model. How small can you get it? Does combining techniques improve the size-accuracy tradeoff?

2. **Sparsity Deep Dive**: Try extreme sparsity (initial=0.7, final=0.95). At what sparsity does accuracy collapse? Plot sparsity vs. accuracy.

3. **Representative Dataset Size**: For integer quantization, try using 10, 50, 100, and 500 representative images. How does calibration dataset size affect INT8 accuracy?

4. **Layer-wise Analysis**: Which layers in EfficientNet contribute most to model size? Use `model.summary()` to identify them. Could you prune convolutional layers differently than dense layers?

## (10) Converting Models for ESP32 Deployment

Our microcontrollers cannot directly load a `.tflite` file. Instead, we convert it to a C header file containing a byte array representation of the model:

```bash
xxd -i int_quant_model.tflite > int_quant_esp.h
```

> **Important:** As you discovered in the feasibility analysis above, even the integer-quantized EfficientNet model is still too large for ESP32's flash memory. This conversion step is shown for reference -- you'll use this exact workflow in **Lab 3** with a much smaller model (a decision tree at ~10 KB) that actually fits.

### ESP32 Setup (for future labs)

1. Open Arduino IDE and install the ESP32 board manager: `Tools -> Board -> Boards Manager -> ESP32 by Espressif Systems`
2. In the Library Manager, search and install **TensorFlowLite_ESP32**

## References:

### [1] https://learnopencv.com/tensorflow-lite-model-optimization-for-on-device-machine-learning/
### [2] https://www.tensorflow.org/model_optimization/guide/pruning/pruning_with_keras
### [3] https://colab.research.google.com/github/tensorflow/model-optimization/blob/master/tensorflow_model_optimization/g3doc/guide/pruning/comprehensive_guide.ipynb#scrollTo=lvpH1Hg7ULFz

## Submission Checklist

Submit the following:

- [ ] Completed Jupyter notebook with **all cells executed** (including your implementations)
- [ ] All `YOUR CODE HERE` sections filled in and working:
  - Size utility function (`get_model_size`)
  - Results comparison table
  - Comparison visualization (bar chart + scatter plot)
- [ ] All discussion questions answered in the markdown cells
- [ ] Comparison visualization showing model sizes and accuracy-vs-size scatter
- [ ] **Written reflection** (1-2 paragraphs) answering:
  - Which compression method yielded the smallest model?
  - Which maintained the highest accuracy?
  - Do any compressed models fit on ESP32? Why or why not?
  - What would you recommend for deploying a classifier on ESP32?